# WindGenerator — Demo

End-to-end wind audio generation using a DDPM diffusion model trained on mel spectrograms.

**Pipeline:** Gaussian noise → DDPM denoising → mel spectrogram → Griffin-Lim → audio

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

No manual setup required — model weights download automatically from Hugging Face.

## 1. Setup

In [ ]:
!git clone https://github.com/alpercagan/WindGenerator.git /content/WindGenerator 2>/dev/null || \
    (cd /content/WindGenerator && git pull)
!pip install -e /content/WindGenerator -q
!pip install huggingface_hub -q

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch: {torch.__version__}')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Download Model Weights

In [ ]:
from huggingface_hub import hf_hub_download
import os

print('Downloading model weights from Hugging Face...')

CHECKPOINT_PATH = hf_hub_download(
    repo_id='alpercagann/wind-generator',
    filename='ckpt_step_074000.pt'
)
print(f'Checkpoint: {CHECKPOINT_PATH}')

MEL_STATS_PATH = hf_hub_download(
    repo_id='alpercagann/wind-generator',
    filename='mel_stats.json'
)
print(f'Mel stats:  {MEL_STATS_PATH}')

OUTPUT_DIR = '/content/generated'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Ready.')

## 3. Generate Audio

Each clip is generated by:
1. Sampling Gaussian noise `(1, 128, 440)` — a random starting point in spectrogram space
2. Running DDPM denoising for 100 steps — the model iteratively refines the noise toward the learned distribution of wind spectrograms
3. Denormalizing the mel spectrogram
4. Converting to audio via `InverseMelScale` + Griffin-Lim phase reconstruction
5. Crossfading the 5s segment with itself to produce a ~10s clip

**Note:** Output quality varies across clips — this reflects the stochastic nature of diffusion sampling. Some clips will sound more wind-like than others.

In [ ]:
import subprocess, glob
import IPython.display as ipd

NUM_CLIPS  = 5    # number of clips to generate
DDPM_STEPS = 100  # denoising steps (more = better quality, slower)

result = subprocess.run([
    'python', '/content/WindGenerator/scripts/generate_audio.py',
    '--diffusion_ckpt', CHECKPOINT_PATH,
    '--mel_stats',      MEL_STATS_PATH,
    '--output_dir',     OUTPUT_DIR,
    '--num_clips',      str(NUM_CLIPS),
    '--ddpm_steps',     str(DDPM_STEPS),
], capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr)

wav_files = sorted(glob.glob(f'{OUTPUT_DIR}/*.wav'))
print(f'\nGenerated {len(wav_files)} clips:')
for f in wav_files:
    print(f'  {os.path.basename(f)}')
    display(ipd.Audio(f))

## 4. Visualize Generated Spectrogram

The mel spectrogram produced by the diffusion model — this is what the model learned to generate from pure noise.

In [ ]:
import torch, json, sys
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
sys.path.insert(0, '/content/WindGenerator/src')

from diffusers import UNet2DModel, DDPMScheduler

# Load model
ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu')
state_dict = ckpt.get('model', ckpt)
c0 = state_dict['conv_in.weight'].shape[0]
has_resnet1 = any('down_blocks.0' in k and 'resnets.1' in k for k in state_dict)
layers_per_block = 2 if has_resnet1 else 1
n_levels = len([k for k in state_dict if k.startswith('down_blocks.') and
                k.endswith('resnets.0.norm1.weight')])

model = UNet2DModel(
    sample_size=(128, 440), in_channels=1, out_channels=1,
    block_out_channels=(c0, c0*2, c0*4),
    layers_per_block=layers_per_block,
    down_block_types=tuple(['DownBlock2D'] * n_levels),
    up_block_types=tuple(['UpBlock2D'] * n_levels),
).to(device)
model.load_state_dict(state_dict)
model.disable_gradient_checkpointing()
model.eval()
print(f'Model: {sum(p.numel() for p in model.parameters()):,} parameters')

# Generate one mel spectrogram
scheduler = DDPMScheduler(num_train_timesteps=1000)
scheduler.set_timesteps(100)
torch.manual_seed(42)
x = torch.randn(1, 1, 128, 440, device=device)
with torch.no_grad():
    for t in scheduler.timesteps:
        noise_pred = model(x, t).sample
        x = scheduler.step(noise_pred, t, x).prev_sample
generated_mel = x.squeeze(1).squeeze(0).cpu().numpy()  # (128, 440)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

im0 = axes[0].imshow(generated_mel, aspect='auto', origin='lower',
                      cmap='magma', vmin=-1, vmax=1)
axes[0].set_title('Generated mel spectrogram (diffusion model output)', fontsize=11)
axes[0].set_xlabel('Time frames (440 = 5.12s)')
axes[0].set_ylabel('Mel frequency bins (128)')
plt.colorbar(im0, ax=axes[0], label='Normalized value')

# Show real mel from inspect folder
real_path = '/content/WindGenerator/outputs/inspect/mel_0.png'
if os.path.exists(real_path):
    img = plt.imread(real_path)
    axes[1].imshow(img, aspect='auto')
    axes[1].set_title('Real wind recording (from dataset)', fontsize=11)
else:
    axes[1].text(0.5, 0.5, 'Dataset mel not available', ha='center',
                va='center', transform=axes[1].transAxes)
    axes[1].set_title('Real wind recording', fontsize=11)

plt.tight_layout()
plt.show()

## 5. Download Clips

In [ ]:
from google.colab import files

wav_files = sorted(glob.glob(f'{OUTPUT_DIR}/*.wav'))
print(f'Downloading {len(wav_files)} clips...')
for f in wav_files:
    files.download(f)

---
## Notes

**On output quality:** Some clips sound more wind-like than others. The diffusion model samples stochastically — each run starts from a different noise tensor. The model was trained for 74,000 steps on 1,966 wind recordings.

**On the metallic quality:** This is from Griffin-Lim phase reconstruction, not the diffusion model. Griffin-Lim estimates phase iteratively but the result is not perceptually optimal. The learned spectrogram content (frequency distribution, temporal texture) is what the model contributes.

**Varying `DDPM_STEPS`:** More steps = better quality, slower. 50 is fast; 100–200 is better.

**Repository:** [github.com/alpercagan/WindGenerator](https://github.com/alpercagan/WindGenerator)  
**Model weights:** [huggingface.co/alpercagann/wind-generator](https://huggingface.co/alpercagann/wind-generator)